<a href="https://colab.research.google.com/github/speculaas/ai201-project3-takemeter/blob/main/fable5_traces_explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fable-5 Traces Explorer
### AI201 · Project 3 · TakeMeter

Lightweight Colab notebook for **inspecting, sampling, flattening, and exporting** readable subsets of the [Glint-Research/Fable-5-traces](https://huggingface.co/datasets/Glint-Research/Fable-5-traces) dataset (~4.67k rows in `train`).

**Good uses for Colab here:**
- `next(iter(ds))` and pretty-printing one row
- Exporting 10–100 examples to JSON / Markdown
- Flattening `trace` JSONL into event tables
- Converting rows into CSV / Parquet snippets
- Checking which fields could map to API calls

**Colab limits to keep in mind:**
- Free Colab resources are **not guaranteed or unlimited**; limits fluctuate with demand.
- Free notebooks run at most **12 hours**; idle runtimes time out.
- GPU access in free Colab is very limited — **use CPU runtime** for this notebook.
- Colab Pro: 100 compute units/month; Pro+: 600 compute units/month, background execution up to 24 hours.

Do **not** use free Colab for huge full-dataset jobs without checkpointing.

---
**Before you start:** Use a **CPU** runtime (Runtime → Change runtime type → CPU).

## Install & load

In [ ]:
!pip -q install datasets pandas pyarrow

from datasets import load_dataset
import json
import pandas as pd
from itertools import islice
from collections import Counter

print("✅ Dependencies ready")

## Stream the first few rows

Hugging Face `datasets` supports `streaming=True`, so you can iterate examples without downloading the full dataset first.

In [ ]:
ds = load_dataset(
    "Glint-Research/Fable-5-traces",
    split="train",
    streaming=True,
)

rows = list(islice(ds, 5))
print(rows[0].keys())

## Inspect one row compactly

In [ ]:
def preview(value, n=1000):
    if isinstance(value, (dict, list)):
        s = json.dumps(value, indent=2, ensure_ascii=False)
    else:
        s = str(value)
    return s[:n]

row = rows[0]
for k, v in row.items():
    print("=" * 80)
    print(k, type(v))
    print(preview(v, 1500))

## Flatten core fields into a readable table

In [ ]:
def short(x, n=500):
    if x is None:
        return None
    if isinstance(x, (dict, list)):
        x = json.dumps(x, ensure_ascii=False)
    else:
        x = str(x)
    return x[:n]

records = []
for i, r in enumerate(rows):
    md = r.get("metadata") or {}
    records.append({
        "i": i,
        "session_id": r.get("session_id"),
        "model": md.get("model"),
        "thinking_level": md.get("thinking_level"),
        "num_user_messages": r.get("num_user_messages"),
        "num_tool_calls": r.get("num_tool_calls"),
        "prompt_preview": short(r.get("prompt"), 300),
        "messages_preview": short(r.get("messages"), 500),
        "tools_preview": short(r.get("tools"), 500),
        "file_path": r.get("file_path"),
    })

df = pd.DataFrame(records)
df

## Parse the `trace` JSONL field into events

In [ ]:
def parse_trace(trace):
    if trace is None:
        return []
    if isinstance(trace, list):
        return trace
    events = []
    for line in str(trace).splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            events.append(json.loads(line))
        except Exception:
            events.append({"_raw": line})
    return events

events = parse_trace(row.get("trace"))
len(events), events[:3]

## Show event types

In [ ]:
event_types = Counter(
    e.get("type", "_missing") for e in events if isinstance(e, dict)
)
event_types

## Export a readable Markdown file

In [ ]:
def row_to_markdown(r, index=0):
    md = []
    md.append(f"# Fable-5 Trace Example {index}")
    md.append("")
    for key in [
        "session_id", "harness", "sent_at",
        "num_user_messages", "num_tool_calls", "file_path",
    ]:
        md.append(f"- **{key}**: `{r.get(key)}`")
    md.append("\n## Metadata\n")
    md.append("```json")
    md.append(json.dumps(r.get("metadata"), indent=2, ensure_ascii=False))
    md.append("```")
    md.append("\n## Messages\n")
    md.append("```json")
    md.append(json.dumps(r.get("messages"), indent=2, ensure_ascii=False))
    md.append("```")
    md.append("\n## Tools\n")
    md.append("```json")
    md.append(json.dumps(r.get("tools"), indent=2, ensure_ascii=False))
    md.append("```")
    md.append("\n## Trace Events Preview\n")
    ev = parse_trace(r.get("trace"))
    md.append("```json")
    md.append(json.dumps(ev[:20], indent=2, ensure_ascii=False))
    md.append("```")
    return "\n".join(md)

markdown = row_to_markdown(row, 0)
with open("fable5_example_0.md", "w", encoding="utf-8") as f:
    f.write(markdown)
print("wrote fable5_example_0.md")

## Export N examples as JSONL

Adjust `N` as needed (10–100 is typical for readable subsets). Re-opens the stream so earlier cells can be re-run independently.

In [ ]:
ds = load_dataset(
    "Glint-Research/Fable-5-traces",
    split="train",
    streaming=True,
)

N = 100
with open("fable5_first_100_pretty.jsonl", "w", encoding="utf-8") as f:
    for i, r in enumerate(islice(ds, N)):
        out = {
            "i": i,
            "session_id": r.get("session_id"),
            "prompt": r.get("prompt"),
            "messages": r.get("messages"),
            "tools": r.get("tools"),
            "metadata": r.get("metadata"),
            "trace_events": parse_trace(r.get("trace")),
            "file_path": r.get("file_path"),
        }
        f.write(json.dumps(out, ensure_ascii=False) + "\n")

print(f"wrote fable5_first_{N}_pretty.jsonl")

## What to look for in Fable-5 rows

Search nested structures for API-like fields, thinking blocks, tool calls, cache controls, and adapter-transformed messages.

In [ ]:
interesting_keys = [
    "thinking", "reasoning", "reasoning_content",
    "cache_control", "system", "max_tokens",
    "tool_use", "tool_result", "messages", "tools",
    "model", "thinking_level",
]

def search_obj(obj, needle, path=""):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{path}.{k}" if path else k
            if needle.lower() in str(k).lower():
                hits.append((p, v))
            hits.extend(search_obj(v, needle, p))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(search_obj(v, needle, f"{path}[{i}]"))
    elif needle.lower() in str(obj).lower():
        hits.append((path, obj))
    return hits

for needle in interesting_keys:
    hits = search_obj(row, needle)
    if hits:
        print("\n###", needle)
        for p, v in hits[:10]:
            print(p, "=>", str(v)[:300])

## Bottom line

Colab is a good lightweight environment for making Fable-5 traces readable. Use **streaming**, export small sampled subsets, and save artifacts to Drive.

When reverse-engineering Fable-5 JSONL traces, the useful mental model is:
1. API messages are translated to chat-template token IDs
2. Generated tokens / logprobs are captured exactly
3. Trajectory logic decides what is safe to train on

Use the search cell above to determine whether each row stores raw API fields, transformed adapter messages, thinking blocks, trace-level reasoning events, tool calls, cache controls, or only summarized metadata.